In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# אתחול Qubit

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
כאשר Circuit מבוצע על יחידת עיבוד קוונטי (QPU) של IBM&reg;, איפוס מרומז מוכנס בדרך כלל בתחילת ה-Circuit כדי להבטיח שה-Qubit מאותחלים לאפס. הדבר נשלט על ידי דגל `init_qubits`, שהוגדר כ[אפשרות ביצוע פרימיטיב](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2).

עם זאת, תהליך האיפוס אינו מושלם, מה שמוביל לשגיאות הכנת מצב. כדי להקל על השגיאה, המערכת גם מכניסה זמן עיכוב חזרה (או `rep_delay`) בין Circuits. לכל Backend יש `rep_delay` שונה כברירת מחדל, אך בדרך כלל הוא ארוך יותר מ-T1 כדי לאפשר לסביבה לאפס את ה-Qubit. ניתן לשלוח שאילתה על `rep_delay` כברירת מחדל על ידי הרצת `backend.default_rep_delay`.

כל ה-QPU של IBM משתמשים בביצוע בקצב חזרה דינמי, המאפשר לך לשנות את `rep_delay` לכל עבודה. ה-Circuits שאתה מגיש בעבודת פרימיטיב מקובצות יחד לביצוע על ה-QPU. ה-Circuits הללו מבוצעים על ידי איטרציה על ה-Circuits לכל shot שהתבקש; הביצוע הוא עמודה-חכמה על מטריצת Circuits ו-shots, כפי שמוצג בתרשים הבא.

![The first column represents shot 0.  The circuits are run in order from 0 through 3.  The second column represents shot 1.  The circuits are run in order from 0 through 3.  The remaining columns follow the same pattern. ](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "Column-wise execution matrix")

מכיוון ש-`rep_delay` מוכנס בין Circuits, כל shot של הביצוע נתקל בעיכוב זה. לכן, הורדת ה-`rep_delay` מפחיתה את זמן ביצוע ה-QPU הכולל, אך על חשבון שיעור שגיאת הכנת מצב מוגבר, כפי שניתן לראות בתמונה הבאה:

![This image shows that as the `rep_delay` value is lowered, the state preparation error rate increases.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "Repetition delay versus error rate")

הגדרת גם `rep_delay=0` וגם `init_qubits=False` למעשה "ממזגת" את ה-Circuits יחד, מכיוון שה-Qubit יתחילו במצב הסופי מה-shot הקודם.

שים לב שבעוד ש-Circuits בעבודת פרימיטיב מקובצות יחד לביצוע QPU, אין ערובה על סדר ביצוע ה-Circuits מ-PUBs. לכן, גם אם אתה מגיש `pubs=[pub1, pub2]`, אין ערובה שה-Circuits מ-`pub1` יופעלו לפני אלה מ-`pub2`. כמו כן, אין ערובה שה-Circuits מאותה עבודה יפעלו כ-batch יחיד על ה-QPU.

## ציין rep_delay לעבודת פרימיטיב

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## הצעדים הבאים
> **Tip:** - נסה דוגמה במדריך [אלגוריתם האופטימיזציה הקוונטי המקורב (QAOA)](/tutorials/quantum-approximate-optimization-algorithm).
>   - עיין כיצד [להתחיל עם פרימיטיבים.](./get-started-with-primitives)